# Tiền xử lí dữ liệu 


* Bắt đầu - khai báo thư viện 

In [16]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler

* Đọc dữ liệu gốc 

In [17]:
train = pd.read_csv("../../data/train (1).csv")
test = pd.read_csv("../../data/test (2).csv")

print("Dữ liệu gốc:")
print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")

Dữ liệu gốc:
Train shape: (14396, 18)
Test shape: (3600, 17)


# --- GIAI ĐOẠN 1: LÀM SẠCH & SỬA LỖI (CLEANING) ---

In [18]:
def clean_basics(df):
    df = df.copy()
    
    # 1. Sửa lỗi đơn vị Duration (Quan trọng nhất)
    # Quy tắc: < 30 thì là phút -> đổi ra ms. > 30 giữ nguyên.
    mask = df['duration_in min/ms'] < 30
    df.loc[mask, 'duration_in min/ms'] = df.loc[mask, 'duration_in min/ms'] * 60000

    # Dùng Logarit để nén các bài hát quá dài lại, giúp model dễ học hơn
    df['duration_in min/ms'] = np.log1p(df['duration_in min/ms'])
    # -------------------------------
    
    # 2. Xử lý dữ liệu thiếu (Missing Values) - Chiến thuật "Hybrid" tối ưu
    # Cột 'instrumentalness': Điền 0 (Giả định thiếu = có lời hát)
    df['instrumentalness'] = df['instrumentalness'].fillna(0)
    
    # Các cột số còn lại ('Popularity', 'key'): Điền Median (Trung vị)
    # Lấy danh sách cột số
    numeric_cols = ['Popularity', 'key']
    imputer = SimpleImputer(strategy='median')
    df[numeric_cols] = imputer.fit_transform(df[numeric_cols])
    
    # 3. Loại bỏ các cột không dùng được (Text rác)
    # Giữ lại ID ở test để tí nữa merge, nhưng bỏ ở train lúc train
    cols_to_drop = ['Artist Name', 'Track Name'] 
    df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
    
    return df

train_clean = clean_basics(train)
test_clean = clean_basics(test)

# --- GIAI ĐOẠN 2: CHUẨN HÓA (SCALING) ---

Bước này cực quan trọng cho SVM và KNN

Lưu ý: Phải giữ lại cột Class (Target) của train và Id của test trước khi scale

In [19]:
# Tách nhãn và Id ra
y_train = train_clean['Class']
train_features = train_clean.drop(columns=['Class', 'Id'])

test_ids = test_clean['Id']
test_features = test_clean.drop(columns=['Id'])

# Scaling: Dùng StandardScaler (đưa về phân phối chuẩn: mean=0, std=1)
# Fit vào Train, sau đó Transform cả Train và Test (để tránh lộ data - Data Leakage)
scaler = StandardScaler()

# Biến đổi
X_train_scaled = pd.DataFrame(scaler.fit_transform(train_features), columns=train_features.columns)
X_test_scaled = pd.DataFrame(scaler.transform(test_features), columns=test_features.columns)

# --- GIAI ĐOẠN 3: ĐÓNG GÓI SẢN PHẨM ---

In [20]:
# Ghép lại Id và Class để xuất file hoàn chỉnh
# File Train: Cần có Class để model học
final_train = X_train_scaled.copy()
final_train['Class'] = y_train.values # Gán lại nhãn

# File Test: Cần có Id để lát nữa làm file nộp bài (submission)
final_test = X_test_scaled.copy()
final_test['Id'] = test_ids.values # Gán lại Id

# Xuất ra file CSV
final_train.to_csv("../../data/train_processed_final.csv", index=False)
final_test.to_csv("../../data/test_processed_final.csv", index=False)

print("\nĐã xử lý xong! 2 file hàng tuyển đã sẵn sàng:")
print("1. train_processed_final.csv (Dùng để train model)")
print("2. test_processed_final.csv (Dùng để dự đoán)")
print("\nCheck thử 5 dòng đầu file train sau khi xử lý:")
print(final_train.head())


Đã xử lý xong! 2 file hàng tuyển đã sẵn sàng:
1. train_processed_final.csv (Dùng để train model)
2. test_processed_final.csv (Dùng để dự đoán)

Check thử 5 dòng đầu file train sau khi xử lý:
   Popularity  danceability    energy       key  loudness      mode  \
0   -0.436403     -1.263390 -0.535777  1.008372  0.308549 -1.334049   
1    1.306175      1.098986  0.358445  1.671545  0.580657  0.749598   
2   -0.029802      0.247081  0.600012  0.345199  0.445342  0.749598   
3   -1.888552     -0.169809 -1.502046  0.013613 -1.678525  0.749598   
4    0.202542      0.132285  0.485585  0.013613  0.691323 -1.334049   

   speechiness  acousticness  instrumentalness  liveness   valence     tempo  \
0    -0.494173      0.422160         -0.488223 -0.563774 -1.049740  1.006641   
1     0.087129     -0.706136         -0.318108 -0.576333 -0.444231  0.346192   
2    -0.214678     -0.790508          1.819958  0.553950  0.620630  1.261369   
3    -0.575203      2.123611         -0.410799  0.654419  0.0